### **ACID - PART 4b - Correct background illumination**

# --- --- ---

### This notebook uses a saved background function to correct for illumination artifacts. The part 4a notebook can be used to create and save the background function.

# --- --- ---

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/03/17


## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import napari
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name, default_multifile_name
from utils.fov_axis_utils import get_fov_ch_shape
from utils.save_image import tifffile_save_ometiff
from utils.miscellaneous_utils import map_image_to_condition
# from image_processing.calculate_background_function import import_fov, calculate_background_function, calculate_bg_funct_per_condition, get_polyfit_bg_funct_channel



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [2]:
# background function computation strategy - this is the strategy used in part4a notebook for
# calculating the background function. Possible options are: 1,2,3.
# 1 - calculate a  background function per channel using all the fields of view in the dataset which are not flagged
# 2 - calculate a background function per each channel and well using the fields of view in the train set which are not
# flagged and belong to the well
# 3 - calculate a background function per each channel and grid position using the fields of view in the train set which are
# not flagged and belong to the same position (aka grid position) across different wells
background_function_strategy = 1

# indicate the path to the directory storing the background function images
background_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\background"

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from either part4a or part3 notebook
# metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the fields of view
# fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov"
fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov"

# # indicate the path to the directory where outputs will be saved
# NOTE: it the directory does not exist, the pipeline will try to create it
# output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\background"
output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov_proc"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part3 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"

# the timestamp (aka the date) present in the file name of the background function image. This is used for selecting the
# background function image to use for the illumination correction in part4b notebook.

# the following options are possible:
# 0) ONLY POSSIBLE FOR STRATEGY 1 - write the full name (extension included) of the background function image file.
# In this case, the file will be selected based on the name and not on the timestamp.
# 1) write the timestamp (aka the date) as a string (e.g. "20251126"). NOTE: the string must match the exact format
# used for in the file name. For example, if the file name is "251126_plate_layout.csv",
# the timestamp string should be "251126" and not "20251126".
# ALSO NOTE: that the if strategy is 1, only a single file with the timestamp in the name is expected in the
# background function directory. If strategy 2 or 3 are selected, a number of files
# corresponding to the number of wells or grid positions, respectively, are expected.
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved ome.tif files will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
# TECHNICAL NOTE: the use of the timestamp is not strictly necessary. In particular: any string which allows to
# uniquely identify the file(s) to use for the background function can be used.
background_function_timestamp = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1


# # --- parameters used for importing the default background function ---
# Indicate the part of the file name to use to select files into background_directory to be used for selecting the default
# file
default_bg_funct_file_target = ".ome.tif" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_bg_funct_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if background_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
background_from_file_name = False # if False, the date will be extracted from the file's last modified timestamp, if True, from the file name.

# separator - used to split the file name and extract the information token with the date
# if background_from_file_name is True
background_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if background_from_file_name is True
background_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if background_from_file_name is True
background_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
background_default_reverse = True

# assert file number - this indicates whether to assert that the number of files matching the
# default file selection criteria is exactly one.
# If True, an assertion error will be raised if the number of files matching the criteria
# is not exactly one. If False, no assertion will be made and the function will return all the files
# matching the default file selection criteria. It is recommended to set this parameter to True when
# background_function_strategy is 1, since in this case only one background function file is expected to
# be used for the illumination correction. When background_function_strategy is 2 or 3, it is recommended
# to set this
background_default_assert_file_number=True
background_default_number_of_files_expected=1

# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for background illumination correction ---
# channel axis to be passed to background calculating functions in order to calculate background functions per channel
# this is the axis along which the channels are organized in the field of view arrays.
# For example, if the field of view arrays have shape (channels, height, width), the channel axis is 0.
channel_axis = 0

# data type to use for computing the illumination correction - the illumination correction is computed by
# dividing the original field of view by a normalized background function. The data type to use for the field of view
# when computing the division can be set here. It is recommended to use a floating point data type
# (e.g. np.float32) to avoid issues with integer division and to preserve the precision of the background function.
correction_operation_dtype_fov = np.float32

# data type to use for computing the illumination correction - the illumination correction is computed by
# dividing the original field of view by a normalized background function. The data type to use for the background
# function when normalizing the histogram values and when computing the division can be set here. It is recommended
# to use a floating point data type (e.g. np.float32) to avoid issues with integer division and to preserve the
# precision of the background function.
correction_operation_dtype_bg = np.float32

# field of view column name in the metadata dataframe
fov_column_name = "ome_tif_file_name"

# well column name in the metadata dataframe
well_column_name = "well"

# plate column name in the metadata dataframe
plate_column_name = "experiment"

# within well position column name in the metadata dataframe
# this is used to identify the position of the field of view within the well
# 49 fields of view were acquired per each well, in a 7x7 grid
gridpos_column_name = "scene_name"


# --- parameters for saving metadata within the saved fields of view after illumination correction ---
# illumination correction image - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the processed image.
# The entry indicates the date when the background correction was done.
proc_img_meta_date_name = "processing_date_yymmdd"

# illumination correction image - date format in metadata - this is the format to use for indicating
# the date when the background correction was done in the metadata saved within the processed image
proc_img_meta_date_format = '%y%m%d'

# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True



# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"

# channel name separator - this is the separator to use for separating the channel name from the rest of
# the column name to be added to the metadata dataframe.
ch_name_separator = "-"

# illumination correction metadata dataframe - computation date column name - this is the name of the column
# to be added to the metadata dataframe to indicate the day when the illumination correction was computed.
illum_correct_df_date_clm_name = f"illumination{column_name_separator}correction{column_name_separator}date"

# illumination correction metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the illumination correction was computed. This is used for saving the date in the
# metadata dataframe
illum_correct_df_meta_date_format = '%y%m%d'


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# include indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# illumination correction image name - savingword - this is the word to use in the field of view file name
# to indicate that the file is an illumination corrected field of view
fov_illumin_corrected_savingword = 'bg'

# illumination correction image name - file suffix - this is the suffix to use for the field of view file name
# when it is saved after illumination correction.
background_img_file_suffix = ".ome.tif"

# illumination correction image - data type - this is the data type to use for saving the field of view
# after illumination correction
fov_illum_corrected_dtype = np.float32

# illumination correction image - photometric - this is the photometric to use for saving the field of view
# after illumination correction
background_img_photometric = 'minisblack'

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}4b.csv"

# processing metadata dataframe name - date format
metadata_date_format = '%Y%m%d'

# hyperparameters dataframe name - date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}4b.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



In [3]:
# import tifffile

# fov_collection = listdirNHF(fov_directory)
# first_fov_path = os.path.join(fov_directory, fov_collection[0])

# with tifffile.TiffFile(first_fov_path) as tif:
#     # print(tif)
#     # imagej_metadata = tif.pages[0].tags['ImageDescription'].value
#     imagej_metadata = tif.imagej_metadata
#     # first_fov_metadata = tifffile.TiffFile(first_fov_path).pages[0].tags['ImageDescription'].value

# print(imagej_metadata)


### Create output directory and secondary output directory if they don't exist

##### Output directory stores the computed background function
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [4]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 3

Run the following cell.

Don't modify the following cell.

In [5]:

# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df



using 20251205_ACID_metadata_part_2.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [6]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df


,Unnamed: 0,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,...,size_c,size_z,size_y,physical_size_y,size_x,physical_size_x,dims_order,int_well,treatment,is_train
1,1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
2,2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A3,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
3,3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A4,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
5,5,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
6,6,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A7,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,92,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G2,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1
93,93,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G3,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1
95,95,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G5,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1
96,96,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G6,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1


In [7]:
# import target files in the background_directory
background_files = listdirNHF(background_directory,
                              target=default_bg_funct_file_target,
                              exclude=default_bg_funct_file_exclude)

if background_function_strategy == 1:
    
    # check if using the default background function name (the most recently saved)
    if background_function_timestamp==None or background_function_timestamp.lower()=="default" or background_function_timestamp=="":
        bg_funct_file_name = default_file_name(file_list=background_files,
                                                       from_file_name=background_from_file_name,
                                                       directory_path=background_directory,
                                                       separator=background_default_separator,
                                                       date_position=background_default_date_position,
                                                       date_format=background_default_date_format,
                                                       reverse=background_default_reverse)
        print(f"using {bg_funct_file_name} as default background function file")
    
    else:
        # select the background function file with the timestamp indicated in background_function_timestamp
        bg_funct_file_name = [f for f in background_files if background_function_timestamp in f][0]

    # open the background function
    background_function = tifffile.imread(os.path.join(background_directory, bg_funct_file_name))


elif background_function_strategy in [2, 3]:
    # check if using the default background function name (the most recently saved)
    if background_function_timestamp==None or background_function_timestamp.lower()=="default" or background_function_timestamp=="":
        bg_funct_file_names = default_multifile_name(file_list=background_files,
                                                       from_file_name=background_from_file_name,
                                                       directory_path=background_directory,
                                                       separator=background_default_separator,
                                                       date_position=background_default_date_position,
                                                       date_format=background_default_date_format,
                                                       reverse=background_default_reverse)
        
        print(f"using the following files as default background function files:")
        for f in bg_funct_file_names:
            print(f)
    else:
        # select the background function files with the timestamp indicated in background_function_timestamp
        bg_funct_file_names = [f for f in background_files if background_function_timestamp in f]


    if background_function_strategy == 2:
        # get unique wells from the metadata dataframe
        unique_conditions = metadata_df[well_column_name].unique()
    elif background_function_strategy == 3:
        # get unique grid positions from the metadata dataframe
        unique_conditions = metadata_df[gridpos_column_name].unique()

    # assert that the number of background function files matches the number of unique wells
    assert len(bg_funct_file_names) == len(unique_conditions), f"The number of background function files ({len(bg_funct_file_names)}) does not match the number of unique conditions ({len(unique_conditions)}). Please check the background function directory and the metadata dataframe."
    
    # map the background function files to the unique conditions (wells or grid positions depending on the strategy)
    bg_funct_file_dict = map_image_to_condition(condition_list=unique_conditions,
                                                image_file_names=bg_funct_file_names,
                                                assert_file_number=background_default_assert_file_number,
                                                number_of_files_expected=background_default_number_of_files_expected)


else:
    raise ValueError(f"Invalid background function strategy: {background_function_strategy}. Please select either 1, 2 or 3.")



using 20260318_ACID_background.ome.tif as default background function file


#### Get the shape and the number of channels of the fields of view - NOTE: it is assumed that all fields of view in the dataset have the same shape and number of channels

Run the following cell.

Don't modify the following cell.

In [ ]:
# # get the shape of the individual fields of view, the number of channels and the shape of individual channels
# fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(metadata_df,
#                                                               fov_directory,
#                                                               fov_clm=fov_column_name,
#                                                               channel_axis=channel_axis,
#                                                               null_value=null_value)


#### Strategy 1 - Calculate a background function by averaging all the fields of view in the dataset

The following cell:
1) import all the fields of view in the dataset
2) stack all the fields of view in a single array.
3) average all the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [ ]:
# # import all fields of view in the dataset and stack them into a single array
# container_arr = import_fov(df=metadata_df_ok,
#                            fov_dir=fov_directory,
#                            fov_clm=fov_column_name,
#                            fov_shape=fov_shape)

# # calculate the background function for individual channels by mean/median average projection
# background_function = calculate_background_function(container_arr=container_arr,
#                                                     method=background_function_method,
#                                                     axis=axis_calc_bg,
#                                                     verbose=verbose_calc_bg)



#### Visualize the background functions per each channel

Run the following cell.

Don't modify the following cell.

In [ ]:
# # istanziate a napari viewer and add the background function to the viewer
# napari_viewer = napari.Viewer()

# for ch in range(background_function.shape[channel_axis]):
#     napari_viewer.add_image(background_function.take(ch, axis=channel_axis), name=f"background_funct_ch_{ch}")



#### Strategy 2 - Calculate a background function per each well of the dataset

The following cell, iteratively per each well:
    
1) import all the fields of view in the dataset belonging to the well
2) stack the fields of view in a single array.
3) average the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [ ]:

# # get the background functions of all the wells in the dataframe and map them into a dictionary
# background_functions_per_well = calculate_bg_funct_per_condition(df=metadata_df_ok,
#                                                                  condition_clm=well_column_name,
#                                                                  fov_dir=fov_directory,
#                                                                  fov_clm=fov_column_name,
#                                                                  fov_shape=fov_shape,
#                                                                  method=background_function_method,
#                                                                  stack_axis=axis_calc_bg,
#                                                                  verbose=verbose_calc_bg)


#### Visualize the background functions per each well and target channels

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# # indicate the channel to visualize
# ch_to_plot = 3

# # --- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---

# # istanziate a napari viewer and add the background function to the viewer
# napari_viewer_1 = napari.Viewer()

# for wel_l in background_functions_per_well:
#     napari_viewer_1.add_image(background_functions_per_well[wel_l].take(ch_to_plot, axis=channel_axis), name=f"background_funct_{wel_l}_{ch_to_plot}")



#### Strategy 3 - Calculate a background function per each field of view grid position

Per each well 49 positions are aquired from a 7x7 grid. The following cells analyse the result of computing a background function per each of the 49 position, by averaging across experiemnts, plates and wells.

Precisely, the following cell, iteratively per each grid position:
    
1) import all the fields of view in the dataset belonging to the grid position
2) stack the fields of view in a single array.
3) average the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [ ]:

# # get the background functions of all the wells in the dataframe and map them into a dictionary
# background_functions_per_gridpos = calculate_bg_funct_per_condition(df=metadata_df_ok,
#                                                                     condition_clm=gridpos_column_name,
#                                                                     fov_dir=fov_directory,
#                                                                     fov_clm=fov_column_name,
#                                                                     fov_shape=fov_shape,
#                                                                     method=background_function_method,
#                                                                     stack_axis=axis_calc_bg,
#                                                                     verbose=verbose_calc_bg)


#### Visualize the background functions per each grid position and target channels

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# # indicate the channel to visualize
# ch_to_plot_1 = 3

# # --- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---

# # istanziate a napari viewer and add the background function to the viewer
# napari_viewer_2 = napari.Viewer()

# for grid_pos in background_functions_per_gridpos:
#     napari_viewer_2.add_image(background_functions_per_gridpos[grid_pos].take(ch_to_plot_1, axis=channel_axis), name=f"background_funct_{grid_pos}_{ch_to_plot_1}")


#### Define the strategy to use for background function calculation

Run the following cell.

MODIFY the following cell.

In [ ]:
# # indicate which strategy to use for creating the background function by fitting a
# # polynomial surface to the averaged background function
# background_function_strategy = 1 # pick between 1, 2 or 3


#### Fit polynomial surface to background function. Update and save the metadata_df. NOTE: the update is done on the metadata_df after train data filter and before filtering of the flagged rows.

Run the following cell.

Don't modify the following cell.

In [ ]:

# # create the background function by fitting a polynomial surface to the averaged background function and
# # according to the strategy indicated above, save the background function as an image
# if background_function_strategy == 1:

#     # create the background function by fitting a polynomial surface to the averaged background function
#     # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up to
#     # the maximum kx and ky indicated will be considered
#     polyfit_background_function = get_polyfit_bg_funct_channel(background_function=background_function,
#                                                                channel_axis=channel_axis,
#                                                                kx= polynomial_order_x,
#                                                                ky= polynomial_order_y,
#                                                                verbose= verbose_calc_bg)

#     # create the background function image name
#     background_img_name = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{background_img_file_suffix}"
    
#     # create the background function image metadata dictionary
#     background_img_metadata_dict = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
#                                     background_img_meta_project_name: project_name,
#                                     background_img_meta_method_name: background_function_method,
#                                     background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}"} # NOTE: order is hardcoded as None
    
#     # save the background function as an image
#     tifffile_save_ometiff(os.path.join(output_directory, background_img_name),
#                                   data=polyfit_background_function.astype(background_img_dtype),
#                                   imagej=save_imagej_compatible,
#                                   photometric=background_img_photometric,
#                                   metadata=background_img_metadata_dict)
    

# elif background_function_strategy == 2:

#     # iterate over the wells
#     for wel_l in background_functions_per_well:

#         # create the background function by fitting a polynomial surface to the averaged background function of the well
#         # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up to the maximum
#         # kx and ky indicated will be considered
#         polyfit_background_function_well = get_polyfit_bg_funct_channel(background_function=background_functions_per_well[wel_l],
#                                                                                    channel_axis=channel_axis,
#                                                                                    kx= polynomial_order_x,
#                                                                                    ky= polynomial_order_y,
#                                                                                    verbose= verbose_calc_bg)

#         # create the background function image name for the well
#         background_img_name_well = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{save_file_name_separator}{wel_l}{background_img_file_suffix}"
        
#         # create the background function image metadata dictionary for the well
#         background_img_metadata_dict_well = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
#                                             background_img_meta_project_name: project_name,
#                                             background_img_meta_method_name: background_function_method,
#                                             background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}", # NOTE: order is hardcoded as None
#                                             "well": wel_l}
        
#         # save the background function as an image for the well
#         tifffile_save_ometiff(os.path.join(output_directory, background_img_name_well),
#                               data=polyfit_background_function_well.astype(background_img_dtype),
#                               imagej=save_imagej_compatible,
#                               photometric=background_img_photometric,
#                               metadata=background_img_metadata_dict_well)

# elif background_function_strategy == 3:

#     # iterate over the grid positions
#     for grid_pos in background_functions_per_gridpos:

#         # create the background function by fitting a polynomial surface to the averaged background function of the grid position
#         # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up
#         # to the maximum kx and ky indicated will be considered
#         polyfit_background_function_gridpos = get_polyfit_bg_funct_channel(background_function=background_functions_per_gridpos[grid_pos],
#                                                                                    channel_axis=channel_axis,
#                                                                                    kx= polynomial_order_x,
#                                                                                    ky= polynomial_order_y,
#                                                                                    verbose= verbose_calc_bg)

#         # create the background function image name for the grid position
#         background_img_name_gridpos = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{save_file_name_separator}{grid_pos}{background_img_file_suffix}"

#         # create the background function image metadata dictionary for the grid position
#         background_img_metadata_dict_gridpos = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
#                                                background_img_meta_project_name: project_name,
#                                                background_img_meta_method_name: background_function_method,
#                                                background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}", # NOTE: order is hardcoded as None
#                                                "grid_position": grid_pos}

#         # save the background function as an image for the grid position
#         tifffile_save_ometiff(os.path.join(output_directory, background_img_name_gridpos),
#                               data=polyfit_background_function_gridpos.astype(background_img_dtype),
#                               imagej=save_imagej_compatible,
#                               photometric=background_img_photometric,
#                               metadata=background_img_metadata_dict_gridpos)

# else:
#     raise ValueError("background_function_strategy must be 1, 2 or 3")


# # --- --- --- UPDATE THE METADATA DATAFRAME WITH THE BACKGROUND FUNCTION INFORMATION --- --- ---
# # add the background function method, polynomial degree and calculation date information to the metadata dataframe
# # as new columns
# # NOTE: the metadata_df used is the one after selecting the train set but before selecting only the non-flagged
# # fields of view
# metadata_df[background_df_date_clm_name] = datetime.datetime.now().strftime(background_df_meta_date_format)
# metadata_df[background_df_method_clm_name] = background_function_method
# for ch_pos, ch_order in enumerate(zip(polynomial_order_x, polynomial_order_y)):
#     metadata_df[f"{background_df_poly_order_x_clm_name}{ch_name_separator}{ch_pos}"] = ch_order[0]
#     metadata_df[f"{background_df_poly_order_y_clm_name}{ch_name_separator}{ch_pos}"] = ch_order[1]

# # save the updated metadata dataframe as a csv file
# metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{metadata_file_suffix}"
# metadata_df.to_csv(os.path.join(output_directory, metadata_df_name), index=save_csv_index)



### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # # collect hyperparameters in a dictionary

# hyperparameter_dict = {

# 'metadata_directory': metadata_directory,
# 'fov_directory': fov_directory,
# 'output_directory': output_directory,
# 'metadata_file_name': metadata_file_name,
# 'is_train_column': is_train_column,
# 'train_val': train_val,
# 'flag_column': flag_column,
# 'flag_value': flag_value,
# 'default_metadata_file_target': default_metadata_file_target,
# 'default_metadata_file_exclude': default_metadata_file_exclude,
# 'metadata_from_file_name': metadata_from_file_name,
# 'metadata_default_separator': metadata_default_separator,
# 'metadata_default_date_position': metadata_default_date_position,
# 'metadata_default_date_format': metadata_default_date_format,
# 'metadata_default_reverse': metadata_default_reverse,
# 'background_function_method': background_function_method,
# 'polynomial_order_x': polynomial_order_x,
# 'polynomial_order_y': polynomial_order_y,
# 'channel_axis': channel_axis,
# 'fov_column_name': fov_column_name,
# 'well_column_name': well_column_name,
# 'plate_column_name': plate_column_name,
# 'gridpos_column_name': gridpos_column_name,
# 'null_value': null_value,
# 'axis_calc_bg': axis_calc_bg,
# 'verbose_calc_bg': verbose_calc_bg,
# 'background_img_meta_date_name': background_img_meta_date_name,
# 'background_img_meta_date_format': background_img_meta_date_format,
# 'background_img_meta_project_name': background_img_meta_project_name,
# 'background_img_meta_method_name': background_img_meta_method_name,
# 'background_img_meta_poly_degree_name': background_img_meta_poly_degree_name,
# 'save_imagej_compatible': save_imagej_compatible,
# 'column_name_separator': column_name_separator,
# 'ch_name_separator': ch_name_separator,
# 'background_df_method_clm_name': background_df_method_clm_name,
# 'background_df_poly_order_x_clm_name': background_df_poly_order_x_clm_name,
# 'background_df_poly_order_y_clm_name': background_df_poly_order_y_clm_name,
# 'background_df_date_clm_name': background_df_date_clm_name,
# 'background_df_meta_date_format': background_df_meta_date_format,
# 'save_file_name_separator': save_file_name_separator,
# 'project_name': project_name,
# 'save_csv_index': save_csv_index,
# 'background_img_name_date_format': background_img_name_date_format,
# 'background_img_savingword': background_img_savingword,
# 'background_img_file_suffix': background_img_file_suffix,
# 'background_img_dtype': background_img_dtype,
# 'background_img_photometric': background_img_photometric,
# 'metadata_savingword': metadata_savingword,
# 'metadata_file_suffix': metadata_file_suffix,
# 'metadata_date_format': metadata_date_format,
# 'hyperparameters_date_format': hyperparameters_date_format,
# 'hyperparameters_savingword': hyperparameters_savingword,
# 'hyperparameters_file_suffix': hyperparameters_file_suffix,
# 'secondary_output_directory': secondary_output_directory,
# 'exist_ok': exist_ok

# }

# # transform the hyperparameter_dict in a pandas series
# hyperparameter_series = pd.Series(hyperparameter_dict)

# # save hyperparamters
# hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
# hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

